In [10]:
import numpy as np
from LanzaModels import TVL1_1D
from ADMMsRustici import MyBackTrackingSolver
from signalClass import *
import time

In [11]:
np.random.seed(24102000)
n = 128

construct blur matrix

In [12]:
#blur matrix construction

a = 0.25
b = 0.5
c = 0.25

diagB = b * np.ones(shape=(n,))
offDiagA = a * np.ones(shape=(n-1,))
offDiagC = c * np.ones(shape=(n-1,))
A = np.diag(diagB, 0) + np.diag(offDiagC, 1) + np.diag(offDiagA, -1)

#apply anti-reflexive BCs

A[0][0] = 2 * a + b
A[0][1] = c - a
A[n-1][n-2] = a - c
A[n-1][n-1] = b + 2 * c

#end blur matrix construction

construct signal

In [13]:
#begin signal construction

PwSignal = signal(n)
RndSignal = signal(n)
sigma = 0.01

PwSignal.generate_cartoon_sign(2, 15)
RndSignal.generate_GG_realization(0, sigma, 1)

xTrue = PwSignal.get_image()
xCorrupted = (A @ xTrue) + RndSignal.get_image()

#end signal construction

Define the TVL2 model

In [14]:
mu = 0.5
VarModel = TVL1_1D.TVL1_1DClass(A, xCorrupted, mu)

Lfid = mu * np.max(np.linalg.svdvals(A)) * np.sqrt(n)
Lreg = np.sqrt(n)
Lphi = np.sqrt(Lfid**2 + Lreg**2)

Now, we need to initialize and define the solver

In [15]:
#begin solver construction
np.random.seed(24102001)

xk = np.random.randn(n,)
yk = np.random.randn(n,)
betak = 1
lk = np.zeros(n)

x0 = xk.copy()
y0 = yk.copy()

MySolver = MyBackTrackingSolver.MyBacktrackingSolverClass(VarModel, xk, yk, lk, betak, Lphi)

#end solver construction

In [16]:
iters = 15

XsolutionHistory = np.zeros(shape=(iters, n))
YsolutionHistory = np.zeros(shape=(iters, n))

lambdaHistory = np.zeros(shape=(iters, n))

betaHistory = np.zeros(shape=(iters,))

PrimalResidueHistory = np.zeros(shape=(iters,))
DualResidueHistory = np.zeros(shape=(iters,))
ImgHistory = np.zeros(shape=(iters,))

CpuTimes = np.zeros(shape=(iters,))

In [17]:
timer = 0

for iter in range(0, iters):

    print(f"{iter + 1} / {iters}")

    sTime = time.perf_counter_ns()

    xk_1, yk_1, lk_1, betak_1, dualResk = MySolver.CallMyIterationStep(xk, yk, lk, betak)

    eTime = time.perf_counter_ns()

    timer += ( (eTime - sTime) / 1e9 )

    
    primalResidue = np.linalg.norm(VarModel.P @ xk_1 + VarModel.Q @ yk_1 - VarModel.c)
    print(f"primalRes: {primalResidue}")

    XsolutionHistory[iter, :] = xk_1
    YsolutionHistory[iter, :] = yk_1
    lambdaHistory[iter, :] = lk_1
    betaHistory[iter] = betak_1

    PrimalResidueHistory[iter] = primalResidue
    DualResidueHistory[iter] = dualResk
    ImgHistory[iter] = VarModel(xk_1, yk_1)
    CpuTimes[iter] = timer

    xk = xk_1
    yk = yk_1
    lk = lk_1
    betak = betak_1

    if (max(primalResidue, dualResk) <= 1e-9):
        print(iter)
        break


1 / 15
iter Mnon: 6
delta L: 0.7036612724862903
iter Backtrack: 65
DualRes: 3.212429320007826e-08
Diamk: 13.214979269098377
primalRes: 0.9159992768164573
2 / 15
iter Mnon: 17
delta L: 0.11380761856939126
iter Backtrack: 19
DualRes: 3.155467754534559e-08
Diamk: 7.837671122168272
primalRes: 0.26504875772735803
3 / 15
iter Mnon: 15
delta L: 0.019055524202861385
iter Backtrack: 44
DualRes: 7.629137079128288e-08
Diamk: 4.898484502256623
primalRes: 0.11985429120250311
4 / 15
iter Mnon: 17
delta L: 0.0032603004283133785
iter Backtrack: 64
DualRes: 1.3662962789693991e-07
Diamk: 3.0061358036834065
primalRes: 0.05291821568017713
5 / 15
iter Mnon: 36
delta L: 0.0007667080397313342
iter Backtrack: 82
DualRes: 2.3225719744730977e-07
Diamk: 1.9179492859575014
primalRes: 0.03166340784903343
6 / 15
iter Mnon: 70
delta L: 0.00026793107623301893
iter Backtrack: 213
DualRes: 4.056594218907329e-07
Diamk: 1.2246432321296203
primalRes: 0.01190919549887102
7 / 15
iter Mnon: 116
delta L: 4.2399280008709184e-0

In [18]:
np.savez_compressed(
    "./MyADMMTVL1-Laplace.npz",
    Xs = XsolutionHistory,
    Ys = YsolutionHistory,
    Ls = lambdaHistory,
    Betas = betaHistory,

    PrimalRes = PrimalResidueHistory,
    DualRes = DualResidueHistory,
	IMGs = ImgHistory,
    CpuTimes = CpuTimes,

    xTrue = xTrue,
    xCorrupted = xCorrupted,
	x0 = x0,
	y0 = y0
)